## ✏️ 前言  
* 🧮 数据处理主要使用**pandas==1.5.3**  
* 🔭 可视化部分主要使用**matplotlib==3.7.2**  


### 1. 导包

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager

import warnings
warnings.filterwarnings("ignore")

## 加载字体
font_path = "/home/mw/project/AlimamaFangYuanTiVF-Thin.ttf"
font_manager.fontManager.addfont(font_path)
prop = font_manager.FontProperties(fname=font_path)

## 定义全局使用
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = prop.get_name()
plt.rcParams['axes.unicode_minus'] = False

### 2. 数据读取及预处理

In [2]:
df = pd.read_csv("/home/mw/input/recruitforit5926/IT行业招聘数据.csv")

In [6]:
## 计算平均薪资
def average_salary(tag):
    if tag.endswith("元/天") and "-" not in tag:
        t = tag[:-3]
        avg = float(t)*30/1000
    elif tag.endswith("元/天") and "-" in tag:
        t = tag[:-3].split("-")
        min_, max_ = float(t[0]), float(t[1])
        avg = float((min_+max_)/2)*30/1000
    elif tag.endswith("元/月"):
        avg = float(tag[:-3])/1000
    elif tag.endswith("k") and "-" not in tag:
        avg = float(tag[:-1])
    else:
        if tag.endswith("元/天"):
            tag = tag[:-3]
        t = tag.split("-")
        min_, max_ = float(t[0]), float(t[1][:-1])
        avg = float(min_+max_)/2
    return avg

df["平均月薪"] = df["岗位薪资"].map(lambda x:average_salary(x) if x not in ["薪资面议","面议"] else x)

df["平均月薪"].replace({"薪资面议":None,"面议":None},inplace=True)

df.dropna(axis=0,inplace=True)

df["平均月薪标签"] = pd.cut(df["平均月薪"],[0,5,8,10,15,20,1000],labels=["<5K","5K~8K","8K~10K","10K~15K","15K~20K",">20K"],right=False)

In [7]:
## 提取本科及以上的数据
data = df.query("学历要求 in ['本科','统招本科','硕士','博士','MBA/EMBA']")

## 对学历进行统一划分(“本科”及“研究生”)
data["学历标签"] = data["学历要求"]

data["学历标签"].replace({"统招本科":"本科","硕士":"研究生","博士":"研究生","MBA/EMBA":"研究生"},inplace=True)

## 提取“经验要求”为0-3年间的数据 --> 减少后续学历比较时薪资受到经验丰富的影响
data_ex = data.query("经验要求 in ['1年以下','1-3年']")

data_ex.head()

,岗位一级分类,岗位二级分类,岗位名称,工作地点,工作省份,岗位薪资,企业名称,经验要求,学历要求,岗位技能需求,企业类型,企业融资状况,企业规模,平均月薪,平均月薪标签,学历标签
14,设计,UI设计师,前端开发工程师,兰州,甘肃,8-10k,兰州飞天网景信息产业有限公司,1-3年,本科,Unknown,互联网,Unknown,500-999人,9.0,8K~10K,本科
19,设计,UI设计师,ui设计师,南宁,广西,5-8k,广西南宁一苇杭之电子商务有限公司,1-3年,本科,"交互设计,UI设计,web端,Illustrator,移动端,网页设计,PC端",贸易/进出口,Unknown,100-499人,6.5,5K~8K,本科
33,设计,UI设计师,UI前端设计师,柳州,广西,4-6k,博达软件,1-3年,本科,"HTML5,框架设计,架构设计,界面开发,IT/信息化领域,Photoshop",计算机软件,融资未公开,50-99人,5.0,5K~8K,本科
43,设计,UI设计师,web前端开发工程师,南宁,广西,3-5k,广西慧云信息技术有限公司,1-3年,本科,"前端开发,jQuery,Web开发,TypeScript",IT服务,B轮,50-99人,4.0,<5K,本科
48,设计,UI设计师,产品经理（偏交付类项目建设） (MJ001164),南宁,广西,8-15k,中国—东盟信息港股份有限公司,1-3年,本科,"金融产品,金融行业,产品设计,用户研究,交互设计,产品规划",互联网,融资未公开,500-999人,11.5,10K~15K,本科


### 3. 数据可视化

In [8]:
tmp = pd.pivot_table(data_ex, values="岗位薪资", index="平均月薪标签", columns="学历标签", aggfunc='count')[::-1]

tmp["占比(本科)"] = tmp["本科"].map(lambda x:x/tmp["本科"].sum())
tmp["占比(研究生)"] = tmp["研究生"].map(lambda x:x/tmp["研究生"].sum())
tmp1 = data_ex.groupby("学历标签").agg({"平均月薪":"mean"}).round(4)

In [9]:
fig = plt.figure(figsize=(10,5),dpi=100)

ax1 = plt.subplot2grid((3,11), (2,0), colspan=11)
bar1 = plt.barh(range(2),tmp1["平均月薪"].tolist()[::-1],height=.5,color=["#2f90b9","#007363"],alpha=.8)
ax1.axis("off")
ax1.bar_label(bar1, [str(x*1000)+"元" for x in tmp1["平均月薪"].tolist()[::-1]], padding=2, color="#000")
plt.title("平均月薪的均值",fontsize=14,loc="left")

ax2 = plt.subplot2grid((3,11), (0,0), rowspan=2, colspan=5)
bar2 = plt.barh(tmp.index.tolist(), tmp["占比(本科)"]*-1, height=.5, color="#2f90b9", label="本科")
ax2.axis("off")
ax2.bar_label(bar2, tmp["占比(本科)"].map(lambda x:str(round(x*100,2))+"%"), padding=2.5, color="#2f90b9")
ax2.legend(frameon=False,fontsize=12)

ax3 = plt.subplot2grid((3,11), (0,6), rowspan=2, colspan=5)
bar3 = plt.barh(tmp.index.tolist(), tmp["占比(研究生)"], height=.5, color="#007363", label="研究生")
ax3.tick_params(axis='y', pad=27.5, color="w") 
ax3.set_yticklabels(tmp.index.tolist(), ha="center", fontsize=12)
ax3.set_xticks([])
for spine in ax3.spines.values():  
    spine.set_visible(False)
ax3.bar_label(bar3, tmp["占比(研究生)"].map(lambda x:str(round(x*100,2))+"%"), padding=2.5, color="#007363")
ax3.legend(frameon=False,fontsize=10)

fig.text(0.5, 0.92, '本科与研究生学历要求的岗位平均月薪分布详情', ha='center', fontsize=20, bbox=dict(facecolor='white', alpha=0))
plt.text(-.35,-4.2,"注意：为保证对比的有效性，仅统计岗位工作经验要求为3年以内的数据", fontsize=10)
plt.show()

<Figure size 1000x500 with 3 Axes>

* 在最低学历要求为本科学历的岗位中，较多数岗位的月薪集中在`10K~15K`，占比达到了`32.37%`，其次是`5K~8K`，占比为`19.26%`。  
* 而在最低学历为研究生要求的岗位中，随着月薪的增高，能提供该薪资的岗位数量也在增多，绝大多数岗位提供的薪资为`>20K`，占比达到`51.29%`。  

* 在所有岗位中，最低学历要求为研究生学历的岗位所提供月薪的平均值为`22163.9`元，相较于本科学历的平均值(`13125.5元`)有着很大的提高。

In [10]:
tmp = pd.pivot_table(data_ex, values="平均月薪", index="企业规模", columns="学历标签", aggfunc='mean')

tmp.drop("Unknown", inplace=True)

orders = ["1-49人","50-99人","100-499人","500-999人","1000-2000人","2000-5000人","5000-10000人","10000人以上"]

tmp = tmp.reindex(orders)  

tmp["占比(本科)"] = tmp["本科"].map(lambda x:x/tmp["本科"].sum())
tmp["占比(研究生)"] = tmp["研究生"].map(lambda x:x/tmp["研究生"].sum())
tmp["考研后增幅"] = tmp["占比(研究生)"] - tmp["占比(本科)"]

In [11]:
fig = plt.figure(figsize=(12,6),dpi=100)

total_width, n =.55, 2
width = total_width / n
x = np.arange(len(tmp.index))

ax1 = plt.subplot2grid((3,1), (0,0), rowspan=1)
ax1.vlines(x, ymin=0, ymax=tmp["考研后增幅"], color='#ed5a65', linewidth=1, zorder=1)
ax1.scatter(x, tmp["考研后增幅"], s=25, color='#ed5a65', zorder=2, label="考研后增幅")
ax1.axhline(y=0,ls="-",color="black",linewidth=1)
ax1.axis("off")
ax1.legend(frameon=False,fontsize=12)

for a,b in zip(x,tmp["考研后增幅"]):
    if b>0 :
        ax1.text(a-.15,b+.005,"+"+str(round(b*100,2))+"%",color="#ed5a65")
    else:
        ax1.text(a-.15,b-.01,str(round(b*100,2))+"%",color="#ed5a65")
plt.title("本科与研究生学历要求的岗位所属公司的规模分布详情",fontsize=20)

ax2 = plt.subplot2grid((3,1), (1,0), rowspan=2)
bar1 = ax2.bar(x - width/2, tmp["占比(本科)"], width=width, label="本科", fc="#5698c3")
bar2 = ax2.bar(x + width/2, tmp["占比(研究生)"], width=width, label="研究生", fc="#68b88e")

for spine in ax2.spines.values():  
    spine.set_visible(False)  
ax2.spines['bottom'].set_visible(True)
plt.xticks(x,tmp.index,fontsize=10,alpha=0.7)
plt.yticks([])
for a,b in zip(x-width,tmp["占比(本科)"]):
    plt.text(a-.15,b+.001,str(round(b*100,2))+"%",color="#5698c3")
for a,b in zip(x-width,tmp["占比(研究生)"]):
    plt.text(a+.3,b+.001,str(round(b*100,2))+"%",color="#68b88e")
ax2.legend(ncol=2,loc="upper left",frameon=False,fontsize=12)
plt.show()

<Figure size 1200x600 with 2 Axes>

* 学历为本科与研究生要求的岗位所属的企业规模数量相差不大，占比差距最大的`1-49人`规模的企业。

In [12]:
## 统一企业融资标签
financ_map = {
    "融资阶段":["融资未公开",'A轮',"B轮",'C轮',"D轮",'D轮及以上',"天使轮"],
    "上市阶段":["沪深A股上市","新三板上市","战略融资",'已上市','创业板上市','港股上市','科创板上市','美股上市','其他股票市场上市'],
    "投资/融资类型":["战略投资","'融资租赁/保理'"],
    "不需要融资":['不需要融资'],
    "其他":["其他"]
    }

def dividing(tag):
    for k,v in financ_map.items():
        if tag in v:
            return k

data_ex["企业融资标签"] = data_ex["企业融资状况"].map(lambda x:dividing(x) if x != "Unknown" else x)

In [13]:
tmp = pd.pivot_table(data_ex, values="平均月薪", index="企业融资标签", columns="学历标签", aggfunc='mean')

def draw_lines(p1, p2, color='black'):
    ax = plt.gca()
    l = mpl.lines.Line2D([p1[0], p2[0]], [p1[1], p2[1]], color='#000', alpha=.2, linewidth=2, zorder=0)
    ax.add_line(l)

    return l

fig, ax = plt.subplots(1, 1, figsize=(8,6), dpi=100)

ax.scatter(tmp["本科"],range(len(tmp.index)), color="#5698c3", s=100, label="本科", zorder=2)
ax.scatter(tmp["研究生"],range(len(tmp.index)),color="#68b88e", s=100, label="研究生", zorder=2)


for i, p1, p2 in zip(df.index, tmp["本科"], tmp["研究生"]):
    draw_lines([p1, i], [p2, i]) 

for i in range(len(tmp.index)):
    if tmp.index[i] != "融资阶段":
        plt.annotate(str(round(tmp["本科"][i],2))+"k", (tmp["本科"][i]-.8, range(len(tmp.index))[i]+.12),alpha=.7)
    else:
        plt.annotate(str(round(tmp["本科"][i],2))+"k", (tmp["本科"][i]-.8, range(len(tmp.index))[i]-.28),alpha=.7)
for i in range(len(tmp.index)):
    if tmp.index[i] != "融资阶段":
        plt.annotate(str(round(tmp["研究生"][i],2))+"k", (tmp["研究生"][i]-.8, range(len(tmp.index))[i]+.12),alpha=.7)
    else:
        plt.annotate(str(round(tmp["研究生"][i],2))+"k", (tmp["研究生"][i]-.8, range(len(tmp.index))[i]-.28),alpha=.7)

ax.set_xticklabels([])
ax.set_yticks(range(len(tmp.index)), tmp.index, fontsize=12)
ax.tick_params(color="w") 
ax.grid(axis="x", alpha=0.9, linestyle='--')
for spine in ax.spines.values():  
    spine.set_visible(False)  
ax.legend(ncol=1,loc="upper right",frameon=False,fontsize=12)
plt.title("本科与研究生学历要求的岗位所属公司的融资情况分布详情",fontsize=20)
plt.show()

<Figure size 800x600 with 1 Axes>

* 对于不同融资情况的公司，均出现了研究生学历岗位的平均月薪高于本科学历的情况。  
* `不需要融资`的企业在两个学历的平均月薪中相较于其他类型的企业均能给出最高的薪资，其研究生相较于本科的薪资差距也是最大的。

In [11]:
tmp = pd.pivot_table(data_ex, values="平均月薪", index="工作省份", columns="学历标签", aggfunc=['mean','count'])

tmp['counts'] = tmp["count"].sum(axis=1)

tmp.drop('count',axis=1,inplace=True)

tmp = tmp[tmp.counts > 500]

tmp["考研后增幅"] = round(tmp["mean"]["研究生"] - tmp["mean"]["本科"],2)

tmp = tmp.sort_values("考研后增幅")

In [12]:
fig = plt.figure(figsize=(8,14),dpi=100)

ax1 = plt.subplot2grid((1,6), (0,0), colspan=4)
total_width, n =.6, 2
width = total_width / n
x = np.arange(len(tmp.index))

ax1.barh(x - width/2, tmp["mean"]["本科"], height=.3, color="#1677b3", label="本科平均月薪")
ax1.barh(x + width/2, tmp["mean"]["研究生"], height=.3, color="#14D5A6", label="研究生平均月薪")

for i in range(len(tmp.index)):
    plt.annotate(str(round(tmp["mean"]["本科"][i],2))+"k", (tmp["mean"]["本科"][i]+.3, i-.3),fontsize=8, color="#1677b3")
for i in range(len(tmp.index)):
    plt.annotate(str(round(tmp["mean"]["研究生"][i],2))+"k", (tmp["mean"]["研究生"][i]+.3, i+.1),fontsize=8, color="#14D5A6")

ax1.set_xlim(0,40)
ax1.set_xticks([])
ax1.set_ylim(-.5,25.5)
ax1.set_yticks(x, tmp.index)
ax1.tick_params(axis='y', color="w") 
ax1.legend(ncol=1,loc="lower right",frameon=False,fontsize=12)
for spine in ax1.spines.values():  
    spine.set_visible(False) 

ax2 = plt.subplot2grid((1,6), (0,4), colspan=1)
ax2.scatter([.2 for i in x], x, s=tmp["counts"]/10, color="#2D813C", alpha=.9, label="岗位数量")

for i in range(len(tmp.index)):
    plt.annotate(str(tmp["counts"][i])+"岗", (.38, i-.1), fontsize=9, color="#2D813C")

ax2.set_xlim(0,1)
ax2.set_xticks([])
ax2.set_ylim(-.5,25.5)
ax2.set_yticks([])
ax2.legend(ncol=1,frameon=False,fontsize=12, bbox_to_anchor=(-.75, 0.095))
for spine in ax2.spines.values():  
    spine.set_visible(False) 

ax3 = plt.subplot2grid((1,6), (0,5), colspan=1)
ax3.scatter([.3 for i in x], x, s=200, marker="^", color="#ee3f4d", label="考研后增幅")

for i in range(len(tmp.index)):
    plt.annotate(str(round(tmp["考研后增幅"][i],2))+"K", (.55, i-.1), fontsize=9, color="#2D813C")

ax3.set_xlim(0,1)
ax3.set_xticks([])
ax3.set_ylim(-.5,25.5)
ax3.set_yticks([])
ax3.legend(ncol=1,frameon=False,fontsize=12, bbox_to_anchor=(-1.78, 0.12))
for spine in ax3.spines.values():  
    spine.set_visible(False) 

plt.text(-7,4,'\n'.join("各地区岗位数量分布及学历不同所导致的平均月薪分布详情"),fontsize=20)
plt.show()

<Figure size 800x1400 with 3 Axes>

* 绝大多数地区的岗位中，研究生学历的岗位均比本科学历的岗位拥有更高的薪资，仅有`吉林`地区在考研工作后的薪资降低，但差距不大，仅有`0.29%`。

In [13]:
tmp = pd.pivot_table(data_ex, values="平均月薪", index="岗位一级分类", columns="学历标签", aggfunc=['mean','count'])

tmp["本科岗位占比"] = tmp["count"]["本科"].div(tmp["count"].sum(axis=1))*100

tmp["研究生岗位占比"] = tmp["count"]["研究生"].div(tmp["count"].sum(axis=1))*100

In [14]:
x = range(len(tmp.index))

fig = plt.figure(figsize=(14,5),dpi=100)

ax1 = plt.subplot(121)
ax1.barh(x, tmp["本科岗位占比"]*-1, height=.3, color="#FF8983", label="本科岗位占比")
ax1.barh(x, tmp["研究生岗位占比"]*-1, left=(tmp["本科岗位占比"]*-1), height=.3, color="#f9a633", label="研究生岗位占比")
ax1.set_xlim(-100,0)
ax1.set_ylim(-.5,3.5)
ax1.set_xticks([])
ax1.set_yticks([])
ax1.legend(ncol=2,loc="lower right",frameon=False,fontsize=9)

for i in range(len(tmp.index)):
    plt.annotate(str(round(tmp["本科岗位占比"][i],2))+"%", (-10, i-.05), color="w")
for i in range(len(tmp.index)):
    plt.annotate(str(round(tmp["研究生岗位占比"][i],2))+"%", (-100, i+.2), color="#f9a633")
for spine in ax1.spines.values():  
    spine.set_visible(False)

ax2 = plt.subplot(122)
total_width, n =.6, 2
width = total_width / n
x = np.arange(len(tmp.index))

bar3 = ax2.barh(x - width/2, tmp["mean"]["本科"], height=.3, color="#1677b3", label="本科平均月薪")
bar4 = ax2.barh(x + width/2, tmp["mean"]["研究生"], height=.3, color="#14D5A6", label="研究生平均月薪")
ax2.set_ylim(-.5,3.5)
ax2.legend(ncol=1,loc="lower right",frameon=False,fontsize=9)
ax2.bar_label(bar3, tmp["mean"]["本科"].map(lambda x:str(round(x,2))+"K"), padding=2.5, color="#2f90b9")
ax2.bar_label(bar4, tmp["mean"]["研究生"].map(lambda x:str(round(x,2))+"K"), padding=2.5, color="#14D5A6")
ax2.set_yticks(np.arange(0,4),[i + "岗" for i in tmp.index],fontsize=20, ha="center")
ax2.tick_params(axis='y', pad=30.5, color="w") 
ax2.set_xticks([])
for spine in ax2.spines.values():  
    spine.set_visible(False)

plt.show()

<Figure size 1400x500 with 2 Axes>

* 技术岗的薪资水平是所有类型的岗位中最高的，本科学历及研究生学历均出现该情况。  
* 所有类型的岗位中，最低学历要求为本科的岗位数量远多于研究生学历。

In [15]:
tmp = pd.pivot_table(data, values="平均月薪", index="经验要求", columns="学历标签", aggfunc='mean')

orders = ["经验不限","实习","应届","1年以下","1-3年","3-5年","5-10年","10年以上"]

tmp = tmp.reindex(orders)  

In [16]:
xtickslabel = tmp.index.tolist()
xtickslabel.insert(0, "1年以下\n(研究生)")

datay = tmp["本科"].tolist()
datay.insert(0, tmp["研究生"].loc["1年以下"])

fig = plt.figure(figsize=(10,5),dpi=100)
ax = plt.subplot(111)

plt.bar(0, datay[0], width=.5, color="#68b88e", label="研究生平均月薪")
plt.bar(range(1,len(tmp.index)+1), tmp["本科"], width=.5, color="#5698c3", label="本科平均月薪")
plt.axhline(y = datay[0], color = '#000', linestyle = '--', alpha=.5, linewidth=.5)
plt.xlim(-.5,len(tmp.index)+.5)
plt.xticks(range(0,len(tmp.index)+1), xtickslabel)
plt.yticks([])
plt.tick_params(axis='x', color="w") 
for i in range(len(tmp.index)+1):
    plt.annotate(str(round(datay[i],2))+"K", (i-.25, datay[i]+.5), color="#000", alpha=.5)
for spine in ax.spines.values():  
    spine.set_visible(False)

plt.legend(ncol=2,loc="upper center",frameon=False,fontsize=12)
plt.title("与研究生毕业1年内平均月薪最为相近的本科岗位要求的学历分布",fontsize=20)
plt.show()

<Figure size 1000x500 with 1 Axes>

* 对于研究生毕业1年内的岗位，本科毕业的人员要有`3-5年`的工作经验才能接近于研究生毕业1年内的薪资水平。  
* 整体上薪资会随着工作经验的增加为增多。  

***  
项目使用数据👉[IT行业招聘数据](https://www.heywhale.com/mw/dataset/659e4dfb8a61ea2f44362af7)  

欢迎各位**Fork**、**点赞**